# Phase 4 Project: Hybrid Movie Recommendation System

## GROUP 2

## Authors: 
1. Andrew Nyakiba

2. Angela Wachira

3. Bobbin Bodo

4. Mercy Chepkoech

5. Ted Mwenda

### Instructor name: MARYANNE MWIKALI


### Project Type: Recommendation System

### Datasets Used: ratings.csv, movies.csv, tags.csv

This notebook develops a hybrid recommendation system using the MovieLens dataset. It combines collaborative filtering and content-based filtering to generate personalized movie recommendations while also addressing the cold start problem for new users.


# 1. PROJECT OVERVIEW

This project aims to build a personalized movie recommendation system using collaborative filtering on the MovieLens dataset. The system will learn from individual users' explicit ratings to predict which unwatched movies they are most likely to enjoy and should return the top 5 personalized recommendations per user.


# 2. BUSINESS PROBLEM

### 2.1 Stakeholder

The primary stakeholder for this project is a movie streaming platform, Ziki, that wants to improve the quality of user recommendations. Ziki is a fictional Nairobi-based mobile streaming startup targeting the East African market. 


### 2.2 BUSINESS PROBLEM

Streaming platforms offer huge catalogs, but without strong personalization, users face decision fatigue. Ziki offers a large catalog of movies, which can make it difficult for users to quickly find content that matches their preferences, leading to decision fatigue and reduced engagement. The goal of this project is to improve user experience by building a recommendation system that provides the top five personalized movie suggestions based on user ratings and content features.

### 2.3 PRIMARY PROJECT OBJECTIVE

The main objective of this project is to build a hybrid recommendation system that can provide top 5 movie recommendations to a user based on prior preferences. 


### 2.4 Secondary Objectives

To support our recommendation system, this analysis seeks to tackle  the following objectives:

1. To identify top genres with highest rates.

2. To develop a user-interface.

3. To build a collaborative filtering model for user-ratings analysis.

4. To build a content based filtering model for descriptive analysis, and to handle the "cold start" problem.



### 2.5 Key Questions

1. What are the top rated genres?

2. Why a Hybrid System?

3. Why work with the 3 chosen datasets?

4. Are there specific trends or patterns in the movie ratings over time?


### 2.6 Why a Hybrid System?

A purely collaborative model works well when enough user-rating history exists, but it performs poorly for new users and less-rated movies. A hybrid system is therefore more appropriate because it combines:

**Collaborative filtering**  for users with interaction history

**Content-based filtering** using movie genres and user-generated tags for cold start situations














### Importation of Libraries

In [2]:

# Uncomment if scikit-surprise is not installed in your environment
# !pip install scikit-surprise

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)


### 3. Loading data

In [3]:
# Aesthetics
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 6)

# Load the datasets
movies  = pd.read_csv('../Data/ml-latest-small/movies.csv')
ratings = pd.read_csv('../Data/ml-latest-small/ratings.csv')
tags    = pd.read_csv('../Data/ml-latest-small/tags.csv')

print("Datasets loaded successfully.")
print(f"Movies: {movies.shape}, Ratings: {ratings.shape}, Tags: {tags.shape}")


Datasets loaded successfully.
Movies: (9742, 3), Ratings: (100836, 4), Tags: (3683, 4)


In [4]:
# First, merge ratings and movies
df = ratings.merge(movies, on='movieId', how='left')
df

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...,...
100831,610,166534,4.0,1493848402,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,1493850091,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,1494273047,Get Out (2017),Horror
100834,610,168252,5.0,1493846352,Logan (2017),Action|Sci-Fi


In [5]:
merged = ratings.merge(movies, on="movieId")
merged.to_csv("merged_data.csv", index=False)
# For Tableau

## 4. DATA UNDERSTANDING

### 4.1 Missing Value Analysis

In [6]:
print("Missing values in ratings:")
print(ratings.isna().sum(), end="\n\n")

print("Missing values in movies:")
print(movies.isna().sum(), end="\n\n")

print("Missing values in tags:")
print(tags.isna().sum())

Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Missing values in movies:
movieId    0
title      0
genres     0
dtype: int64

Missing values in tags:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


### 4.2 Duplicate Values Analysis

In [7]:
# Check duplicates
print("Duplicate rows in ratings:", ratings.duplicated().sum())
print("Duplicate rows in movies:", movies.duplicated().sum())
print("Duplicate rows in tags:", tags.duplicated().sum())


Duplicate rows in ratings: 0
Duplicate rows in movies: 0
Duplicate rows in tags: 0


### 4.3 Global statistics

In [8]:
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)
sparsity = 1 - (n_ratings / (n_users * n_movies))

print(f"Unique Users    : {n_users}")
print(f"Unique Movies   : {n_movies}")
print(f"Total Ratings   : {n_ratings}")
print(f"Matrix Sparsity : {sparsity:.2%}")

Unique Users    : 610
Unique Movies   : 9724
Total Ratings   : 100836
Matrix Sparsity : 98.30%


## 5. DATA CLEANING AND PRE-PROCESSING 

In [9]:
# Drop timestamp column
df = df.drop(columns=['timestamp'])

# Confirm
df.head()

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


## 5.1 Data Filtering (Denoising)
To ensure model reliability, we filter for movies with at least 10 ratings and users who have rated at least 5 movies.

In [ ]:
# Create movie_ratings dataset (merge + drop unnecessary columns)
movie_ratings = pd.merge(ratings, movies, on='movieId')

In [15]:
# 1. Filter Movies: Keep those with 10 or more ratings
movie_counts = movie_ratings.groupby('movieId').size()
popular_movies = movie_counts[movie_counts >= 5].index
df_filtered = movie_ratings[movie_ratings['movieId'].isin(popular_movies)]

# 2. Filter Users: Keep those with 5 or more ratings
user_counts = df_filtered.groupby('userId').size()
active_users = user_counts[user_counts >=5 ].index
df_filtered = df_filtered[df_filtered['userId'].isin(active_users)]

print(f"Original records: {len(movie_ratings)}")
print(f"Filtered records: {len(df_filtered)}")
print(f"Movies kept: {df_filtered['movieId'].nunique()}")
print(f"Users kept: {df_filtered['userId'].nunique()}")

Original records: 100836
Filtered records: 90274
Movies kept: 3650
Users kept: 610
